In [ ]:
import os
import csv
import numpy as np
import torch
import pandas as pd
from Cyclist_env_RDA_2nano import cyclist_env
from scipy.fft import fft, ifft
from scipy.linalg import dft

OUTPUT_DIR = "./learn_dataset_fixed_angle"
os.makedirs(OUTPUT_DIR, exist_ok=True)

NUM_SAMPLES = 300
SEED = 42
rng = np.random.default_rng(SEED)

In [2]:
def Radar_setting():
    ###### symbol time & carrier frequency ######
    T_symbol = 5.575 * 1e-6              # symbol duration, with CP time
    T_OFDM = 5.2125 * 1e-6
    f_carrier = 28 * 1e+9
    Tc = 2.545*1e-9                      # sampling time

    ###### tx/rx ######
    N_ant = 16                       # the number of antennas
    # BW = 1.966080e+9                      # chirp bandwidth
    BW = 400*1e+6                    # chirp bandwidth
    BW_sub = BW/N_ant
    N_sample = int(np.floor(T_symbol/Tc))           # the number of samples of single chirp
    rx_sample = 813 + N_sample

    ###### Radar setting ######

    mu = BW_sub/T_symbol * 0.98
    l_speed = 299792458

    lam = l_speed/f_carrier
    env = cyclist_env(f_carrier, N_ant, BW, BW_sub, N_sample, rx_sample, Tc, mu, l_speed)
    return T_symbol,f_carrier,Tc,N_ant,BW,BW_sub,N_sample,rx_sample, mu, l_speed, lam, env

In [3]:
def physical_to_rd_indices(range_m, velocity_mps, N_trans, sym_duration, lam, Tc, range_offset=620, num_range_bins=190):
    c = 299792458.0
    vel_res = lam / (2.0 * sym_duration * N_trans)
    range_res = c * Tc / 2.0

    d_idx = int(np.round(velocity_mps / vel_res)) % N_trans
    r_idx = int(np.round(range_m / range_res - range_offset))

    valid = (0 <= d_idx < N_trans) and (0 <= r_idx < num_range_bins)
    return d_idx, r_idx, valid

In [4]:
def syntax_check(num, idx, y_value, velocity_value, x_pos):
    idx_list = np.zeros(num, dtype=np.int32)
    position, velocity = np.zeros((num, 3)), np.zeros((num, 3))
    i=0
    s = set()
    while i < num:
        # 初期値における距離間隔を保証するための処理
        if idx+10 in s or idx-10 in s or idx in s: continue
        for j in range(-20,20): s.add(idx+j)
        
        idx_list[i] = idx
        position[i] = np.array([x_pos[idx], y_value, 1])
        # v_cy[i] = np.array([np.random.rand(1)[0]*5.56, 0, 0])
        velocity[i] = np.array([velocity_value, 0, 0])
        
        i+=1
    return idx_list, position, velocity

In [14]:
def get_snapshot_data(cy_idx_target, ve_idx_target,x_pos, p_bs, tx, cy_v_value, ve_v_value, num_cy, num_ve, num_rp, sym_duration, N_trans,Tc, env):
    # Cyclistについて
    cy_idx = np.zeros(num_cy, dtype=np.int32)
    p_cy, v_cy = np.zeros((num_cy, 3)), np.zeros((num_cy, 3))
    # 範囲外アクセスを防ぐための処理
    cy_idx, p_cy, v_cy = syntax_check(num_cy, cy_idx_target, 15, cy_v_value, x_pos)
    n_cy = len(p_cy)
    
    # Vehicleについて
    ve_idx = np.zeros(num_ve, dtype=np.int32)
    p_ve, v_ve = np.zeros((num_ve, 3)), np.zeros((num_ve, 3))
    # 範囲外アクセスを防ぐための処理
    ve_idx, p_ve, v_ve = syntax_check(num_ve, ve_idx_target, 7.5, ve_v_value, x_pos)
    n_ve = len(p_ve)

    # ramppostについて
    p_rp, v_rp = np.zeros((num_rp, 3)), np.zeros((num_rp, 3))
    n_rp = len(p_rp)
    
    # csvファイルの読み込み
    cy_idx += 1
    ve_idx += 1

    P_rx_cy, tstemp_rx_cy, phase_cy, P_rx_ve, tstemp_rx_ve, phase_ve, P_rx_rp, tstemp_rx_rp, phase_rp = [], [], [], [], [], [], [], [], []
    # 3番目のデータから読み込み開始（最初の数個はノイズのみのため）
    for i in range(len(cy_idx)):
        df = pd.read_csv("./ped/2.545nano/far/complex-impulse-response-Run{:04d}-Sensor_0_Tx_0_to_Rx_0.csv".format(cy_idx[i]))
        tstemp_rx_cy.append(np.round((df["Time (s)"]/Tc)).values[3:])
        P_rx_cy.append(df["| Total Complex Impulse Response total | (W)"].values[3:])
        phase_cy.append(df['Phase( Total Complex Impulse Response total ) (rad)'].values[3:])
    for i in range(len(ve_idx)):
        df = pd.read_csv("./vehicle/2.545nano/far/complex-impulse-response-Run{:04d}-Sensor_0_Tx_0_to_Rx_0.csv".format(ve_idx[i]))
        tstemp_rx_ve.append(np.round((df["Time (s)"].values[3:]/Tc)))
        P_rx_ve.append(df["| Total Complex Impulse Response total | (W)"].values[3:])
        phase_ve.append(df['Phase( Total Complex Impulse Response total ) (rad)'].values[3:])
    # 2.545のほうを使う！
    for i in range(2,3):
        df = pd.read_csv("./Ramposts/2.545nano/"+str(i)+"/complex-impulse-response-Run0001-Sensor_0_Tx_0_to_Rx_0.csv")
        tstemp_rx_rp.append(np.round((df["Time (s)"].values[5:]/Tc)))
        P_rx_rp.append(df["| Total Complex Impulse Response total | (W)"].values[5:])
        phase_rp.append(df['Phase( Total Complex Impulse Response total ) (rad)'].values[5:])
    phys_quantities = env.phys_quantities(p_bs, p_cy, v_cy, n_cy, p_ve, v_ve, n_ve, p_rp, v_rp, n_rp)
    #print(phys_quantities["relative_velocity"]["cyclists"])
    real_cy_coordinates = []
    real_vel_coordinates = []
    real_rp_coordinates = []

    for i in range(len(p_cy)):
        real_cy_coordinates.append(np.array([p_cy[i][0], p_cy[i][1]]))
    for i in range(len(p_ve)):
        real_vel_coordinates.append(np.array([p_ve[i][0], p_ve[i][1]]))
    for i in range(len(p_rp)):
        real_rp_coordinates.append(np.array([p_rp[i][0], p_rp[i][1]]))
    
        
    ######### We need some functions here. it is like this. read the csv. this outputs the time stemp and corresponding rx value.
    # P_rx_cy_dB = -114
    #P_rx_cy_dB = -111
    #P_rx_ve_dB = -114
    #P_rx_cy_dB = 0
    #P_rx_ve_dB = 0
    
    P_N_dB = -87.98

    Y = env.rx_multiple(tx, P_rx_cy, tstemp_rx_cy, phase_cy, P_rx_ve, tstemp_rx_ve, phase_ve, P_rx_rp, tstemp_rx_rp, phase_rp, P_N_dB, sym_duration, N_trans)
    
    return Y, phys_quantities, real_cy_coordinates, real_vel_coordinates, real_rp_coordinates

In [6]:
def stevec(N_ant, angle):
    # 入力された角度に対するステアリングベクトルの計算
    resp = (np.arange(N_ant)-N_ant*0.5+0.5).reshape([-1,1])
    resp = np.exp(1j*resp*np.pi*np.sin(angle))
    return np.matrix(resp)

In [7]:
# 固定角度グリッドの設定
# 基地局とセンシング範囲が約300m離れているため走査範囲を±5°に絞る
N_FIXED = 10
FIXED_ANGLES = np.linspace(-5, 5, N_FIXED)  # 単位: 度

In [8]:
def MUSIC_method_improved(Y, N_ant, num_signals, frame_count):
    '''
    修正前
    Y_music = np.mean(Y, axis=0)
    R_yy = Y_music@np.conjugate(np.transpose(Y_music))
    '''
    # 修正後: 
    # チャープ軸(0)とサンプル軸(2)の両方を「スナップショット」として扱います。
    # (N_trans, N_ant, N_sample) -> (N_ant, N_trans * N_sample) に変形
    # これにより、N_trans(チャープ数)が増えるほど、スナップショット数が増え、
    # 共分散行列 R_yy の推定精度が向上します。
    N_trans, N_ant_data, N_sample = Y.shape
    # データを (アンテナ数 x 全サンプル数) の2次元行列に変形
    Y_music = Y.transpose(1, 0, 2).reshape(N_ant_data, -1)
    
    # 共分散行列の計算
    # (行列サイズは アンテナ数 x アンテナ数 のまま変わらないので計算負荷は軽微です)
    R_yy = (Y_music @ np.conjugate(Y_music.T)) / Y_music.shape[1]
    
    # 1.固有値分解
    eig_val, eig_vec = np.linalg.eig(R_yy)
    
    # 2.固有値をソートして雑音部分空間を取得
    idx = np.abs(eig_val).argsort()[::-1]
    # インデックスによって固有値と固有ベクトルの対応関係を維持しつつソート
    eig_val = eig_val[idx]
    eig_vec = eig_vec[:, idx]
    # 雑音部分空間Uを取得

    # num_signalsを固有値から推定?
    
    # デバッグ用
    num_signals = 5
    
    U = eig_vec[:, num_signals:]
    

    # 3.雑音部分空間とvalごとのステアリングベクトルの内積を計算
    resp = []
    argm = (np.arange(1800)-900)/100
    for val in argm:
        # ステアリングベクトルの計算
        stv = stevec(N_ant, val*np.pi/180)
        # 雑音部分空間との内積
        p = stv.T@U
        # L2ノルムの二乗を計算(pp^H)
        pp = p*np.conjugate(np.transpose(p))
        # A1は行列を1次元配列に変換するメソッド
        # respに逆数を追加
        # 分子は角度推定の上では不要なので省略している
        resp.append(1/(np.abs(pp)**2).A1)
    
    # 4.ピーク検出
    M = np.max(resp)
    
    # デバッグ用
    #MUSIC_method_Debug(eig_val, N_ant, num_signals, resp, argm, M, frame_count)
    
    est_ang = []
    # 三点比較によるピーク検出
    a, b = resp[0][0], resp[1][0]
    for i in range(1,len(argm)-1):
        c = resp[i+1][0]
        #if a < b and b > c and b > 0.2 * M:
        if a < b and b > c and b > 0.00001 * M:
            est_ang.append(argm[i])
        a, b = b, c
    return est_ang

In [9]:
def FFT(N_trans, N_ant, rx_sample, Y, tx, N_sample, est_ang, sym_duration, lam, Tc, env):
    # ドップラー処理
    Dopp_dft = dft(N_trans)/np.sqrt(N_trans)
    Y_dft = np.zeros((N_trans, N_ant, rx_sample), dtype=np.complex128)
    for i in range(N_ant):
        Y_dft[:,i,:] = Dopp_dft@Y[:,i,:]
    
    final_objects_all = []
    rd_maps = []

    P_range = np.arange(190)+620

    # FFTによる位置推定処理
    tx_arr = np.array(tx) 
    n_fft = 2**(int(rx_sample + N_sample).bit_length())
    #n_fft = 2048
    
    # 窓関数を掛けずにFFTする
    TX_F_all = fft(tx_arr, n=n_fft, axis=1) 
    Y_F_all = fft(Y_dft, n=n_fft, axis=2)
    
    for i in range(len(est_ang)):
        current_angle = est_ang[i]
        ang_rad = current_angle * np.pi / 180
        
        # ステアリングベクトル (N_ant, 1)
        s_vec = np.array(stevec(N_ant, ang_rad))
        
        # --- 周波数領域でのビームフォーミング ---
        
        # 受信信号の合成: Y_combined = sum(Y * s)
        # s_vec を (1, N_ant, 1) に変形して放送
        # Y_F_all: (N_trans, N_ant, n_fft)
        # 重み付け和をとってアンテナ次元を潰す -> (N_trans, n_fft)
        # 修正前
        w_vec = np.conjugate(s_vec).reshape(1, N_ant, 1)
        Y_F_combined = np.sum(Y_F_all * w_vec, axis=1)
        # 修正後
        w_vec = s_vec.reshape(1, N_ant, 1) 
        Y_F_combined = np.sum(Y_F_all * w_vec, axis=1)

        # 送信信号(レプリカ)の合成: X_combined = sum(X * s*)
        # 元の数式 g = s* s^H X より、Xにかかる重みは s^H (つまり sの共役)
        # TX_F_all: (N_ant, n_fft)
        s_vec_conj_reshape = np.conjugate(s_vec).reshape(N_ant, 1)
        TX_F_combined = np.sum(TX_F_all * s_vec_conj_reshape, axis=0) # -> (n_fft,)

        # --- 相関演算 (Correlation) ---
        # Correlation = IFFT( FFT(Y) * conj(FFT(X)) )
        # Broadcasting: (N_trans, n_fft) * (n_fft,)
        CORR_f = Y_F_combined * np.conjugate(TX_F_combined)
        
        # 時間領域に戻す
        corr_time = ifft(CORR_f, axis=1)

        # --- 必要な範囲を切り出し ---
        rdresp_single = corr_time[:, P_range]

        # --- 結果保存 ---
        rd_maps.append(rdresp_single.copy())
        rd_maps.append(i)

    return rd_maps

In [10]:
def generate_one_sample(sample_id, cy_target_idx, ve_target_idx):
    T_symbol, f_carrier, Tc, N_ant, BW, BW_sub, N_sample, rx_sample, mu, l_speed, lam, env = Radar_setting()
    N_chirp = 89
    N_trans = N_chirp
    tx = env.tx()
    p_bs = np.array([250, -18, 50])
    x_pos = list(np.arange(-450, -149) / 10)
    num_cy, num_ve, num_rp = 1, 1, 0
    cy_idx_target = int(cy_target_idx)
    ve_idx_target = int(ve_target_idx)
    cy_v_value, ve_v_value = 6, 10

    Y, phys_quantities, real_cy_coordinates, real_vel_coordinates, real_rp_coordinates = get_snapshot_data(
        cy_idx_target, ve_idx_target, x_pos, p_bs, tx,
        cy_v_value, ve_v_value, num_cy, num_ve, num_rp,
        T_symbol, N_trans, Tc, env
    )

    # MUSICによる角度推定を廃止し、固定角度グリッドを使用
    rd_maps = FFT(N_trans, N_ant, rx_sample, Y, tx, N_sample, FIXED_ANGLES, T_symbol, lam, Tc, env)

    cy_true_range = float(np.asarray(phys_quantities["range"]["cyclists"][0]).reshape(-1)[0])
    cy_true_velocity = float(np.asarray(phys_quantities["relative_velocity"]["cyclists"][0]).reshape(-1)[0])
    cy_true_angle_deg = float(np.degrees(np.asarray(phys_quantities["angle"]["cyclists"][0]).reshape(-1)[0]))
    ve_true_range = float(np.asarray(phys_quantities["range"]["vehicles"][0]).reshape(-1)[0])
    ve_true_velocity = float(np.asarray(phys_quantities["relative_velocity"]["vehicles"][0]).reshape(-1)[0])
    ve_true_angle_deg = float(np.degrees(np.asarray(phys_quantities["angle"]["vehicles"][0]).reshape(-1)[0]))

    cy_d_idx, cy_r_idx, cy_valid = physical_to_rd_indices(
        cy_true_range, cy_true_velocity, N_trans, T_symbol, lam, Tc
    )
    ve_d_idx, ve_r_idx, ve_valid = physical_to_rd_indices(
        ve_true_range, ve_true_velocity, N_trans, T_symbol, lam, Tc
    )
    valid_all = int(cy_valid and ve_valid)

    save_path = os.path.join(OUTPUT_DIR, f"sample_{sample_id:05d}.npz")
    np.savez_compressed(
        save_path,
        rd_maps=np.array(rd_maps[::2], dtype=np.complex64),  # shape: (N_FIXED, N_doppler, N_range)
        fixed_angles=np.array(FIXED_ANGLES, dtype=np.float32),
        cyclist_true_range=np.float32(cy_true_range),
        cyclist_true_velocity=np.float32(cy_true_velocity),
        cyclist_true_angle_deg=np.float32(cy_true_angle_deg),
        cyclist_true_d_idx=np.int32(cy_d_idx),
        cyclist_true_r_idx=np.int32(cy_r_idx),
        vehicle_true_range=np.float32(ve_true_range),
        vehicle_true_velocity=np.float32(ve_true_velocity),
        vehicle_true_angle_deg=np.float32(ve_true_angle_deg),
        vehicle_true_d_idx=np.int32(ve_d_idx),
        vehicle_true_r_idx=np.int32(ve_r_idx),
        valid_cyclist=np.int32(cy_valid),
        valid_vehicle=np.int32(ve_valid),
        valid_all=np.int32(valid_all),
        source_index_cyclist=np.int32(cy_target_idx),
        source_index_vehicle=np.int32(ve_target_idx),
    )
    return {
        "sample_id": sample_id,
        "file": save_path,
        "source_index_cyclist": int(cy_target_idx),
        "source_index_vehicle": int(ve_target_idx),
        "cyclist_true_range": cy_true_range,
        "cyclist_true_velocity": cy_true_velocity,
        "cyclist_true_angle_deg": cy_true_angle_deg,
        "cyclist_true_d_idx": int(cy_d_idx),
        "cyclist_true_r_idx": int(cy_r_idx),
        "vehicle_true_range": ve_true_range,
        "vehicle_true_velocity": ve_true_velocity,
        "vehicle_true_angle_deg": ve_true_angle_deg,
        "vehicle_true_d_idx": int(ve_d_idx),
        "vehicle_true_r_idx": int(ve_r_idx),
        "valid_cyclist": int(cy_valid),
        "valid_vehicle": int(ve_valid),
        "valid_all": int(valid_all),
    }

In [13]:
rows = []

candidate_cy_indices = np.arange(1, 301)  # cyclist の候補
candidate_ve_indices = rng.permutation(np.arange(1, 301))  # vehicle の候補をシャッフル
num_generate = min(NUM_SAMPLES, len(candidate_cy_indices), len(candidate_ve_indices))

for sample_id in range(num_generate):
    cy_target_idx = int(candidate_cy_indices[sample_id])
    ve_target_idx = int(candidate_ve_indices[sample_id])
    row = generate_one_sample(sample_id, cy_target_idx, ve_target_idx)
    rows.append(row)
    if sample_id % 20 == 0:
        print(sample_id, row["file"])

meta_path = os.path.join(OUTPUT_DIR, "metadata.csv")
with open(meta_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)

print("saved:", meta_path)


0 ./learn_dataset_fixed_angle\sample_00000.npz
saved: ./learn_dataset_fixed_angle\metadata.csv
